# Milestone 1 — Tabular MLP for Rating Band Classification

**Task:** predict rating band (low/medium/high) from tabular metadata.
**Baseline:** Logistic Regression. **Deep:** Keras MLP.
**Deliverable:** MLP outperforms baseline + learning curves + error analysis.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, callbacks
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, classification_report)

PROJECT_ROOT = '/content/drive/MyDrive/smart-product-intelligence'

## 1. Load data and build 8 tabular features

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

products = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'products.csv'))
reviews = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'reviews.csv'))

def make_features(df, agg_reviews):
    feats = pd.DataFrame()
    feats['price_filled'] = df['price_clean'].fillna(20.0)
    feats['price_was_missing'] = df['price_clean'].isna().astype(int)
    feats['rating_number'] = df['rating_number'].fillna(0)
    agg = (agg_reviews.groupby('parent_asin')
           .agg(n_reviews=('rating','count'),
                avg_helpful=('helpful_vote','mean'),
                avg_review_len=('text_length','mean')).reset_index())
    merged = df.merge(agg, on='parent_asin', how='left')
    feats['n_reviews_in_sample'] = merged['n_reviews'].fillna(0).values
    feats['avg_helpful'] = merged['avg_helpful'].fillna(0).values
    feats['avg_review_len'] = merged['avg_review_len'].fillna(100).values
    feats['title_len'] = df['title'].astype(str).str.len()
    feats['desc_len'] = df['description_text'].astype(str).str.len()
    return feats.values.astype(np.float32)

def make_target(df):
    return pd.cut(df['average_rating'].fillna(0),
                  bins=[-0.1, 3.5, 4.3, 5.1], labels=[0,1,2]).astype(int).values

train_p = products[products['split']=='train'].reset_index(drop=True)
val_p = products[products['split']=='val'].reset_index(drop=True)
test_p = products[products['split']=='test'].reset_index(drop=True)
train_r = reviews[reviews['split']=='train']
val_r = reviews[reviews['split']=='val']
test_r = reviews[reviews['split']=='test']

X_train = make_features(train_p, train_r); y_train = make_target(train_p)
X_val = make_features(val_p, val_r); y_val = make_target(val_p)
X_test = make_features(test_p, test_r); y_test = make_target(test_p)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## 2. Baseline — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average='macro')
print(f'LogReg: accuracy={acc_lr:.3f}, macro-F1={f1_lr:.3f}')

## 3. Deep model — MLP (8 → 64 → 32 → 16 → 3 with BN + Dropout)

In [ ]:
tf.random.set_seed(42); np.random.seed(42)

model = tf.keras.Sequential([
    layers.Input(shape=(8,)),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax')
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw = dict(enumerate(class_weights))

es = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train_s, y_train, validation_data=(X_val_s, y_val),
                    epochs=100, batch_size=64, class_weight=cw,
                    callbacks=[es], verbose=0)
print(f'Trained {len(history.history["loss"])} epochs (early stopping)')

y_pred_mlp = model.predict(X_test_s, verbose=0).argmax(axis=1)
acc_mlp = accuracy_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp, average='macro')
print(f'\nMLP:    accuracy={acc_mlp:.3f}, macro-F1={f1_mlp:.3f}')
print(f'Improvement: +{100*(f1_mlp-f1_lr)/f1_lr:.1f}% macro-F1')

## 4. Error analysis

In [ ]:
print(classification_report(y_test, y_pred_mlp,
                            target_names=['low','medium','high'], zero_division=0))

## 5. Figures

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=os.path.join(PROJECT_ROOT, 'figures', '01_m1_results.png'))

## 6. Summary

| Model | Accuracy | Macro-F1 |
|---|---|---|
| LogReg (baseline) | 0.465 | 0.372 |
| **MLP** | **0.514** | **0.428** |

MLP improves on baseline by **+15.0% macro-F1**.
The "medium" rating band remains the hardest class for both — a direct
consequence of the dataset's 5-star skew.